# RetNet with Range Images

Use a sample scan in *.bin* format and labels as *.label* from SemanticKitti dataset

In [1]:
import numpy as np
from PIL import Image

### Read point cloud

In [3]:
# files
scan_file = '000000.bin'
label_file = '000000.label'

In [4]:
# read scan which is a vector of (x, y, z, remission)
scan = np.fromfile(scan_file, dtype=np.float32)
scan = scan.reshape((-1, 4))
print(f'Number of points in sample scan: {scan.shape[0]}')

# read labels
labels = np.fromfile(label_file, dtype=np.uint32)
labels = labels.reshape(-1)
print(f'Number of labels in scan: {labels.shape[0]}')
labels = labels & 0xFFFF

assert scan.shape[0] == labels.shape[0], 'Different number of points'

Number of points in sample scan: 123389
Number of labels in scan: 123389


Create colors for labels

In [5]:
# create colors for all labels
max_labels = 300
color_list = np.random.uniform(low=0.0, high=1.0, size=(max_labels, 3))
# force zero to a gray-ish color
color_list[0] = np.full(3, 0.1)

# array of colors, shape Nx3
colors = np.array([color_list[i] for i in labels])
colors = colors.reshape((-1, 3))

assert colors.shape[0] == scan.shape[0]

### Create Range Image

In [6]:
H = 64
W = 1024

fov_up = 3.0 / 180.0 * np.pi
fov_down = -25 / 180.0 * np.pi

In [7]:
# range image with (x, y, z, depth, remission) features
range_image = np.zeros((H, W, 5), np.float32)
label_image = np.zeros((H, W, 1), np.uint32)
img = np.zeros((H, W, 4), np.uint8)

In [8]:
# Compute the range of the point cloud
r = np.sqrt(np.sum(np.power(scan[:, :3], 2), axis=1))

# Compute the polar and azimuthal angles of the points
pitch = np.arcsin(scan[:, 2] / r)
yaw = np.arctan2(scan[:, 1], scan[:, 0])

fov = fov_up + np.abs(fov_down)

In [9]:
# create range image
for i, p in enumerate(scan):
    # print(pitch[i], fov_down, fov)
    u = H * (1 - ((pitch[i] + np.abs(fov_down)) / fov))
    v = W * (0.5 * ((yaw[i] / (np.pi / 2)) + 1.0))

    # round to the nearest integer
    u = int(np.round(min(u, H-1)))
    v = int(np.round(min(v, W-1)))

    u = max(0, u)
    v = max(0, v)

    # print(u, v)

    # range image
    range_image[u, v, :4] = p
    range_image[u, v, 4] = r[i]
    
    # label image
    label_image[u, v] = labels[i]

    # color image
    rgb = 255 * colors[i, :]
    img[u, v, :3] = rgb.astype(int)
    img[u, v, 3] = scan[i, 3]

In [10]:
# convert to PIL image and visualize
image = Image.fromarray(img[:, :, :3])
image.show()

In [ ]:
# np.savetxt('labels.txt', label_image.reshape(-1, 1024), fmt='%d')

### Input Range Image

In [11]:
print('Range image shape: ', range_image.shape)
print('Sample pixel feature: ', range_image[0, 0]) # (x, y, z, remission, depth/range)

Range image shape:  (64, 1024, 5)
Sample pixel feature:  [-3.5873933  -5.838372    0.35274866  0.44        6.8615165 ]


### REM

In [12]:
# TODO

### Patch Embedding

In [13]:
patch_size = 4

In [14]:
# number of patches
num_patches = H * W // (patch_size**2)
print('Number of patches: ', num_patches)

# TODO divide into patches
image_patches = range_image.reshape((num_patches, patch_size, patch_size, range_image.shape[2]))
print('Image pathces size: ', image_patches.shape)

Number of patches:  4096
Image pathces size:  (4096, 4, 4, 5)


In [16]:
# extract a single patch 4x4
patch = range_image[:4, :4, :]
print('Patch shape: ', patch.shape)
print(patch)

Patch shape:  (4, 4, 5)
[[[ -3.5873933   -5.838372     0.35274866   0.44         6.8615165 ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]]

 [[ -0.6929782   -6.983232     0.28763857   0.26         7.023424  ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]]

 [[ -3.5752637   -5.7144995    0.22691695   0.2          6.744591  ]
  [  0.12900202 -69.31769      2.652209     0.          69.36853   ]
  [  0.46013498 -70.65973      2.6987567    0.          70.712746  ]
  [  0.           0.           0.           0.           0.        ]]

 [[ -3.560644    -5.8135505    0.18266638   0.18         6.819745  ]
  [  0.2583011  -70.76301      2.2530925    0.          70.79934   ]
  [ 

In [17]:
# flatten patch
flatten_patch = patch.flatten()
print('Flatten patch shape: ', flatten_patch.shape)
print('Flatten patch: ', flatten_patch)

Flatten patch shape:  (80,)
Flatten patch:  [ -3.5873933   -5.838372     0.35274866   0.44         6.8615165
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
  -0.6929782   -6.983232     0.28763857   0.26         7.023424
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
  -3.5752637   -5.7144995    0.22691695   0.2          6.744591
   0.12900202 -69.31769      2.652209     0.          69.36853
   0.46013498 -70.65973      2.6987567    0.          70.712746
   0.           0.           0.           0.           0.
  -3.560644    -5.8135505    0.18266638   0.18         6.819745
   0.2583011  -70.76301      2.2530925    0.          70.79934
   0.4808866  -70.846054     2.2548654    0.          70.88356
   0.           0.           0.         

In [18]:
# extract a batch of 4 patches
patches = []
for i in range(0, 16, 4):
    tmp_patch = range_image[i:i+4, i:i+4, :]
    flat_patch = tmp_patch.reshape(16, range_image.shape[2])
    patches.append((flat_patch))
    
patches = np.array(patches)[:, :, :4]
print(patches.shape)

(4, 16, 4)


## RetNet

XPOS Implementation from [Microsoft torchscale github](https://github.com/microsoft/torchscale/blob/main/torchscale/component/xpos_relative_position.py)

In [19]:
import math
import torch
import torch.nn as nn

In [29]:
def fixed_pos_embedding(x):
    seq_len, dim = x.shape
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim) / dim))
    sinusoid_inp = (
        torch.einsum("i , j -> i j", torch.arange(0, seq_len, dtype=torch.float), inv_freq).to(x)
    )
    return torch.sin(sinusoid_inp), torch.cos(sinusoid_inp)

def rotate_every_two(x):
    x1 = x[:, :, ::2]
    x2 = x[:, :, 1::2]
    x = torch.stack((-x2, x1), dim=-1)
    return x.flatten(-2)  # in einsum notation: rearrange(x, '... d j -> ... (d j)')\

def duplicate_interleave(m):
    """
    A simple version of `torch.repeat_interleave` for duplicating a matrix while interleaving the copy.
    """
    dim0 = m.shape[0]
    m = m.view(-1, 1)  # flatten the matrix
    m = m.repeat(1, 2)  # repeat all elements into the 2nd dimension
    m = m.view(dim0, -1)  # reshape into a matrix, interleaving the copy
    return m

def apply_rotary_pos_emb(x, sin, cos, scale=1):
    sin, cos = map(lambda t: duplicate_interleave(t * scale), (sin, cos))
    #print(x.shape)
    #print(cos.shape)
    # einsum notation for lambda t: repeat(t[offset:x.shape[1]+offset,:], "n d -> () n () (d j)", j=2)
    return (x * cos) + (rotate_every_two(x) * sin)


class XPOS(nn.Module):
    def __init__(
        self, head_dim, scale_base=512
    ):
        super().__init__()
        self.head_dim = head_dim
        self.scale_base = scale_base
        self.register_buffer(
            "scale", (torch.arange(0, head_dim, 2) + 0.4 * head_dim) / (1.4 * head_dim)
        )

    def forward(self, x, offset=0, downscale=False):
        length = x.shape[1]
        min_pos = 0
        max_pos = length + offset + min_pos
        scale = self.scale ** torch.arange(min_pos, max_pos, 1).to(self.scale).div(self.scale_base)[:, None]
        #print('SCALE = ', scale.shape)
        sin, cos = fixed_pos_embedding(scale)
        #print('SHAPE SIN: ', sin.shape)
        #print('SHAPE COS: ', cos.shape)

        if scale.shape[0] > length:
            scale = scale[-length:]
            sin = sin[-length:]
            cos = cos[-length:]
        
        if downscale:
            scale = 1 / scale

        x = apply_rotary_pos_emb(x, sin, cos, scale)
        return x
    
    def forward_reverse(self, x, offset=0, downscale=False):
        length = x.shape[1]
        min_pos = -(length + offset) // 2
        max_pos = length + offset + min_pos
        scale = self.scale ** torch.arange(min_pos, max_pos, 1).to(self.scale).div(self.scale_base)[:, None]
        sin, cos = fixed_pos_embedding(scale)

        if scale.shape[0] > length:
            scale = scale[-length:]
            sin = sin[-length:]
            cos = cos[-length:]
        
        if downscale:
            scale = 1 / scale

        x = apply_rotary_pos_emb(x, -sin, cos, scale)
        return x
    
# test
xx = torch.eye(4).unsqueeze(0)
#print(xx.shape)
xpos = XPOS(4)
x_rot = xpos(xx)
# apply reverse
x_rot_rev = xpos.forward(xx)

print(x_rot @ x_rot_rev.transpose(-1, -2))

tensor([[[ 1.0000, -0.8394,  0.0000,  0.0000],
         [-0.8394,  0.9951,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.9966, -0.0100],
         [ 0.0000,  0.0000, -0.0100,  0.9948]]])


### Simple Retention

In [30]:
# parameters
num_head = 1
hidden_size = patches.shape[2]
head_size = patches.shape[2]
v_dim = head_size * 2
gamma = (1 - torch.exp(torch.linspace(math.log(1/32), math.log(1/512), num_head))).tolist()

xpos = XPOS(head_size)

groupNorm = nn.GroupNorm(num_head, v_dim)

In [31]:
# weights
Wq = torch.randn(hidden_size, head_size) / hidden_size
Wk = torch.randn(hidden_size, head_size) / hidden_size
Wv = torch.randn(hidden_size, v_dim) / hidden_size

In [32]:
# compute matrix D
def get_D(sequence_length):
    n = torch.arange(sequence_length).unsqueeze(1)
    m = torch.arange(sequence_length).unsqueeze(0)

    # Broadcast self.gamma ** (n - m) with appropriate masking to set values where n < m to 0
    D = (gamma[0] ** (n - m)) * (n >= m).float()  #this results in some NaN when n is much larger than m
    # fill the NaN with 0
    D[D != D] = 0

    return D

Forward Pass (default parallel representation)

In [33]:
x = torch.from_numpy(patches)
print(x.shape) # (batch size, sequence length, hidden size) (num patches, num pixels, hidden size)

torch.Size([4, 16, 4])


In [34]:
D = get_D(x.shape[1])
print(D)

# compute Q and K
Q = x @ Wq
K = x @ Wk

print(Q.shape)

# positional encoding
Q = xpos(Q)
K = xpos(K, downscale=True)

# compute V    
V = x @ Wv

# compute retention
ret = (Q @ K.permute(0, 2, 1)) * D.unsqueeze(0) @ V
print(ret)

# group norm
output = groupNorm(ret.reshape(-1, v_dim)).reshape(ret.shape)

print('RET: ', output)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9688, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9385, 0.9688, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9091, 0.9385, 0.9688, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8807, 0.9091, 0.9385, 0.9688, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8532, 0.8807, 0.9091, 0.9385, 0.9688, 1.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8266, 0.8532, 0.8807, 0.9091, 0.9385, 0.9688, 1.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.00

In [37]:
print(output.shape) # (num patches, pixel in single patch, V feature)

torch.Size([4, 16, 8])


In [38]:
print(Q.shape)
print(K.shape)
print(V.shape)
print(D.shape)
prod = Q @ K.permute(0, 2, 1)
prod = prod * D.unsqueeze(0)
print(prod.shape)
retention = prod @ V
print(retention.shape)

torch.Size([4, 16, 4])
torch.Size([4, 16, 4])
torch.Size([4, 16, 8])
torch.Size([16, 16])
torch.Size([4, 16, 16])
torch.Size([4, 16, 8])


Parallel Representation

In [39]:
# extract a single pixel
x_n = x[:, :1, :]
print(x_n.shape)

torch.Size([4, 1, 4])


In [40]:
sn_1 = torch.rand((4, 1, 8))

In [41]:
# compute Q and K
Qn = x_n @ Wq
Kn = x_n @ Wk

# Xpos
Qn = xpos(Qn, 1)
Kn = xpos(Kn, 1, downscale=True)

# compute V
Vn = x_n @ Wv

s_n = gamma[0] * sn_1 + (K.transpose(-1, -2) @ V)

rec_output = Q @ s_n

print(rec_output)
print(rec_output.shape)

tensor([[[-2.9239e+03, -4.5024e+03, -1.6365e+03, -8.8057e+00,  1.2212e+02,
          -1.3795e+03,  1.7691e+03,  1.5464e+03],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 3.4931e+03,  5.4564e+03,  2.0129e+03, -2.9155e+01, -1.7661e+02,
           1.7355e+03, -2.1386e+03, -1.8663e+03],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],


In [43]:
print(rec_output.shape) # (num of patches, pixel per patch, V features)

torch.Size([4, 16, 8])


MLP layer

In [44]:
# TODO

#### Semantic Head

In [47]:
# matrices for MLP weights
W1 = torch.rand((output.shape[2], output.shape[2] * 2), dtype=torch.float32)
W2 = torch.rand((output.shape[2] * 2, 20), dtype=torch.float32) # 20 as number of classes

In [49]:
y1 = output @ W1
y2 = y1 @ W2

In [50]:
print(y2.shape)

torch.Size([4, 16, 20])


In [51]:
# simulate predictions
pred = torch.argmax(y2, dim=2)

In [56]:
# extract first patch predicitons
patch_pred = pred[0].reshape((4, 4, 1))

patch_labels = label_image[:4, :4, :]

assert patch_pred.shape == patch_labels.shape, 'Predictions and labels have different shape'

print(patch_pred)
print(patch_labels)

tensor([[[16],
         [ 0],
         [ 0],
         [ 0]],

        [[16],
         [ 0],
         [ 0],
         [ 0]],

        [[ 0],
         [19],
         [19],
         [ 0]],

        [[19],
         [ 0],
         [19],
         [ 0]]])
[[[71]
  [ 0]
  [ 0]
  [ 0]]

 [[71]
  [ 0]
  [ 0]
  [ 0]]

 [[71]
  [ 0]
  [ 0]
  [ 0]]

 [[71]
  [ 0]
  [ 0]
  [ 0]]]
